In [ ]:
import os, subprocess, sys

# ── 1. Install uv ────────────────────────────────────────────────────────────
subprocess.run("wget -qO- https://astral.sh/uv/install.sh | sh", shell=True)
os.environ["PATH"] += os.pathsep + os.path.expanduser("~/.local/bin")

# ── 2. Create venv ──────────────────────────────────────────────────────────
subprocess.run("uv venv .venv --seed --clear --python 3.11", shell=True)
pip = ".venv/bin/python -m pip"

# ── 3. Phase 1: Blackwell PyTorch (The Critical Fix) ─────────────────────────
# We use CUDA 12.8 nightly because it contains the sm_120 kernels.
print("Installing Blackwell-ready PyTorch (CUDA 12.8 nightly)...")
subprocess.run(f"""
{pip} install --pre torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/nightly/cu128 --force-reinstall
""", shell=True, check=True)

# ── 4. Phase 2: Math Reasoning Stack ─────────────────────────────────────────
# We install these without version pins for the most part to allow the
# resolver to find Blackwell-compatible wheels (Qwen-2507 needs newer libs).
print("Installing remaining math-reasoning stack...")
subprocess.run(f"""
{pip} install \
    "numpy<2" \
    "transformers>=4.51.0" \
    "accelerate>=0.30" \
    "peft>=0.14.0" \
    "bitsandbytes>=0.45.0" \
    "vllm>=0.7.0" \
    "triton" \
    "nvidia-cuda-runtime-cu12" \
    "nvidia-cuda-cupti-cu12" \
    "nvidia-cudnn-cu12" \
    "nvidia-nvjitlink-cu12" \
    "pandas" \
    "antlr4-python3-runtime==4.11.1" \
    sympy tqdm sentencepiece ipykernel jupyter
""", shell=True, check=True)

# ── 5. Environment & Kernel Setup ───────────────────────────────────────────
venv_site = ".venv/lib/python3.11/site-packages"
ld_paths = [
    f"{venv_site}/nvidia/nvjitlink/lib",
    f"{venv_site}/nvidia/cuda_runtime/lib",
    f"{venv_site}/nvidia/cudnn/lib",
    "/usr/local/cuda/lib64",
    "/usr/local/cuda-12.8/lib64",
]
os.environ["LD_LIBRARY_PATH"] = ":".join(ld_paths) + ":" + os.environ.get("LD_LIBRARY_PATH", "")

# Permanently fix bashrc for background tasks and future sessions
bashrc_line = f'\nexport LD_LIBRARY_PATH={":".join(ld_paths)}:$LD_LIBRARY_PATH\n'
with open(os.path.expanduser("~/.bashrc"), "a") as f:
    f.write(bashrc_line)

subprocess.run(
    ".venv/bin/python -m ipykernel install --user --name cse151bV2 --display-name 'Python (cse151bV2)'",
    shell=True
)
print("\n✅ Environment built for Blackwell (sm_120).")
print("👉 ACTION: Select kernel 'cse151bV2' in the top right → RESTART KERNEL.")

In [1]:
!.venv/bin/python -u train_grpo_blackwell.py \
    --init-adapter data/lora_math_adapter/final_adapter \
    --max-completion 4096 \
    --vllm-mem 0.5 \
    --epochs 3 \
    --data data/public_shorten.jsonl


[20:48:30] STEP 1 / 6  CUDA sanity check (bf16, L40 48GB profile)
CUDA not available.
